# Training of GReaT (GPT-2) Model on the Australian Credit Approval Dataset

## Introduction

This notebook continues the workflow from "Training of Generative Models on the Australian Credit Approval Dataset" (https://colab.research.google.com/drive/1Q7mmTqwbMvrkbbyl9nJKjOP_4xIgPoUY#scrollTo=EU2dvJNATSRW), focusing specifically on the training of the final generative model, GReaT with the full GPT-2 backbone.

The separation of this notebook from the others is motivated by computational requirements: transformer-based models demand significantly more resources and time compared to the other generative models. Training GReaT independently ensures modularity, reproducibility, and manageable execution within the workflow.

The resulting trained model will then be used to generate synthetic data for subsequent evaluation.

## 0 Imports and function definition

We import the necessary libraries for data handling, training the GReaT model, and saving the trained object. This includes pandas for data manipulation, GReaT for transformer-based generative modeling, and pickle for model persistence.

In [ ]:
!pip install be_great peft==0.9.0 sdv

  Using cached be_great-0.0.9-py3-none-any.whl.metadata (5.8 kB)
  Using cached peft-0.9.0-py3-none-any.whl.metadata (13 kB)
  Using cached sdv-1.24.1-py3-none-any.whl.metadata (14 kB)
  Using cached boto3-1.39.11-py3-none-any.whl.metadata (6.7 kB)
  Using cached botocore-1.39.11-py3-none-any.whl.metadata (5.7 kB)
  Using cached copulas-0.12.3-py3-none-any.whl.metadata (9.5 kB)
  Using cached ctgan-0.11.0-py3-none-any.whl.metadata (10 kB)
  Using cached deepecho-0.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached rdt-1.17.1-py3-none-any.whl.metadata (10 kB)
  Using cached sdmetrics-0.21.0-py3-none-any.whl.metadata (9.4 kB)
  Using cached jmespath-1.0.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.13.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached faker-37.4.2-py3-none-any.whl.metadata (15 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_

In [ ]:
import pandas as pd
import os
from be_great import GReaT
import pickle

In [ ]:
os.environ["WANDB_DISABLED"] = "true" #to avoid the requests from WANDB

The preprocess_dataset function is used here as in the previous notebooks to standardize data types and handle missing values, preparing the dataset for GReaT training.

In [ ]:
def preprocess_dataset(df, target_column, categorical_features, numerical_features):
    df = df.copy()

    for col in categorical_features:
        df[col] = df[col].astype('object')

    for col in numerical_features:
        df[col] = df[col].astype('float')

    for col in categorical_features:
        df[col] = df[col].fillna('Unknown')

    for col in numerical_features:
        df[col] = df[col].fillna(df[col].mean())

    return df


The train_and_save_model_GReaT function is used to train the GReaT model with the full GPT-2 backbone on the Australian Credit Approval dataset and save the trained model for later use.

In [ ]:
def train_and_save_model_GReaT(df, batch_size, epochs, efficient_finetuning):

  filename = "GReaT_australian.pkl"

  model = GReaT(llm="gpt2", batch_size=batch_size, epochs=epochs, fp16=True, efficient_finetuning=efficient_finetuning)
  model.fit(df)

  # Salvataggio modello
  with open(filename, 'wb') as f:
    pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)

## 1 Dataset loading


The Australian Credit Approval dataset is loaded from its CSV file into a pandas DataFrame, providing a structured format for preprocessing and model training. The dataset can be accessed from the following source: UCI Machine Learning Repository - https://archive.ics.uci.edu/dataset/143/statlog+australian+credit+approval.
Preprocessing follows the same procedure defined in previous notebooks, standardizing feature types and imputing missing values to produce a clean dataset ready for training GReaT.

In [ ]:
#Load Australian Credit Approval
australian_df = pd.read_csv('australian_credit_approval.csv')

In [ ]:
australian_categorical_features = ['A1', 'A4', 'A5', 'A6', 'A8', 'A9', 'A11', 'A12']
australian_numerical_features = ['A2', 'A3', 'A7', 'A10', 'A13', 'A14']

In [ ]:
preprocessed_dataset = preprocess_dataset(australian_df, 'Class', australian_categorical_features, australian_numerical_features)

/tmp/ipython-input-4-1313495981.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna('Unknown')


## 2 Train

Following preprocessing, we proceed to train the GReaT model using the previously defined function. This ensures that the model learns the structure of the Australian Credit Approval dataset and is saved for subsequent synthetic data generation.

### GReaT

In [ ]:
train_and_save_model_GReaT(preprocessed_dataset, 32, 200, None)

/usr/local/lib/python3.11/dist-packages/be_great/great.py:174: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `GReaTTrainer.__init__`. Use `processing_class` instead.
  great_trainer = GReaTTrainer(
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,0.813900
1000,0.701700
1500,0.682400
2000,0.666100
2500,0.648800
3000,0.632700
3500,0.619700
4000,0.609900
